# 2차전지 데이터 분석 · 중급 2일차
## Orange3에서 파이썬으로

---

### 이 노트북의 목적

**파이썬 코드를 외우는 것이 목적이 아닙니다.**

Orange3에서 익힌 판단을 파이썬에서도 할 수 있는지 확인하고,
**AI가 준 코드가 맞는지 검증하는 능력**을 기르는 것이 목적입니다.

| 단계 | 하는 일 |
|---|---|
| **①** | 프롬프트를 복사해 Gemini에게 요청 |
| **②** | 받은 코드를 빈 셀에 붙여넣고 **실행** |
| **③** | Orange3에서 본 숫자와 **대조** |
| **④** | 다르면 원인을 찾기 |

---

### 시작하기 전에

**1. Gemini 켜기** — 왼쪽 아래 ✨ 아이콘을 누르면 채팅창이 열립니다.

**2. 파일 올리기** — 왼쪽 폴더 아이콘 📁 → 업로드

| 파일 | 언제 만들었나 |
|---|---|
| `battery_cycles.csv` | 처음부터 제공된 데이터 |
| `battery_prep.csv` | **M2에서 저장** (Save Data) |
| `battery_rf.pkcls` | **M4에서 저장** (Save Model) |

---

### 중요 — 참고 코드를 먼저 보지 마십시오

각 실습에는 접힌 **참고 코드**가 있습니다.
**먼저 Gemini에게 물어보고, 실행해 보고, 막혔을 때만** 펼치십시오.

정답을 베끼면 아무것도 남지 않습니다.

---
# 0. 환경 준비

먼저 필요한 것들을 불러옵니다. 이 셀은 그대로 실행하십시오.

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# 데이터 불러오기
df = pd.read_csv("battery_cycles.csv")

print("행 · 열 :", df.shape)
print("컬럼    :", list(df.columns))
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'battery_cycles.csv'

**확인** — Orange3의 Data Info에서 본 것과 같아야 합니다.

| 항목 | Orange3에서 본 값 |
|---|---|
| 행 수 | **636** |
| 열 수 | **14** |

다르다면 파일을 잘못 올린 것입니다.

---
# 1. 데이터 이해 — M1 대조

Orange3의 **Column Statistics**에서 본 것을 파이썬으로 확인합니다.

### 실습 1-1 · 컬럼별 통계 확인

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">나는 배터리 SOH 데이터를 분석하고 있어.
battery_cycles.csv 를 pandas 로 읽었고 변수명은 df 야.

다음을 해줘.
  1. 각 컬럼의 평균, 중앙값, 최솟값, 최댓값, 결측 개수를 표로 보여주기
  2. 결측이 있는 컬럼만 따로 알려주기

조건:
- 코드에 한 줄씩 주석을 달아줘
- 나는 파이썬 초보야. 어려운 문법은 피해줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>Orange3 Column Statistics에서 본 값과 비교하십시오.

| 확인할 것 | 값 |
|---|---|
| `soh` 최댓값 | **101.77** |
| 결측이 있는 컬럼 | `re_ohm` · `rct_ohm` |
| 결측 개수 | 각 **57개** |</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
# 컬럼별 기초 통계
stats = df.describe().T[["mean", "50%", "min", "max"]]
stats.columns = ["평균", "중앙값", "최솟값", "최댓값"]

# 결측 개수를 붙인다
stats["결측"] = df.isna().sum()
print(stats.round(4))

# 결측이 있는 컬럼만
missing = df.isna().sum()
print("\n결측이 있는 컬럼:")
print(missing[missing > 0])
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

### 실습 1-2 · 결측이 어디에 있는지

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">battery_cycles.csv 의 re_ohm 컬럼에 결측이 있어.
이 결측이 무작위로 흩어져 있는지, 특정 구간에 몰려 있는지 확인하고 싶어.

다음을 해줘.
  1. 셀(cell_id)별로 결측이 몇 개인지
  2. B0005 에서 결측인 행의 cycle 번호를 출력
  3. 결측이 앞쪽에 몰려 있는지 판단할 수 있게 보여주기

조건: 코드에 주석을 달고, 결과를 보고 어떻게 해석해야 하는지도 알려줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>**M2에서 확인한 것과 같습니까?**

| 셀 | 결측 개수 | 위치 |
|---|---|---|
| B0005 · B0006 · B0007 | 각 **19개** | 사이클 **1~19** |
| B0018 | **0개** | — |

무작위가 아니라 **초반에 몰려 있는 구조적 결측**입니다.</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
# 셀별 결측 개수
print("셀별 re_ohm 결측:")
print(df.groupby("cell_id")["re_ohm"].apply(lambda s: s.isna().sum()))

# B0005 에서 결측인 사이클 번호
b5 = df[df.cell_id == "B0005"]
missing_cycles = b5[b5.re_ohm.isna()]["cycle"].tolist()
print("\nB0005 결측 사이클:", missing_cycles)

# 연속인지 확인
print("최솟값:", min(missing_cycles), "· 최댓값:", max(missing_cycles))
print("개수:", len(missing_cycles), "→ 연속이면 앞쪽에 몰린 구조적 결측")
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

---
# 2. 전처리 — M2 대조

Orange3에서 저장한 **`battery_prep.csv`**를 불러와 확인합니다.

### 저장한 파일이 제대로 넘어왔는지 확인

M2에서 Save Data로 저장한 파일을 읽어 봅니다.

In [ ]:
# M2에서 저장한 전처리 완료 데이터
prep = pd.read_csv("battery_prep.csv")

print("전처리 후 :", prep.shape)
print("결측 개수 :", prep.isna().sum().sum())
prep.head()

**여기서 확인할 것**

Orange3에서 본 행 수와 같아야 합니다. 다르면 저장 과정에서 무언가 빠진 것입니다.

파일이 없다면 이 셀은 건너뛰고 아래로 진행하십시오.

### 실습 2-1 · 결측 처리 방법 비교

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">나는 배터리 데이터의 결측을 어떻게 처리할지 실험하고 있어.
battery_cycles.csv 에서 B0005 셀만 쓸 거야.

입력 변수는 이 9개야.
  cycle, discharge_time, max_temp, mean_temp, mean_voltage,
  min_voltage, re_ohm, rct_ohm, mean_current
목표 변수는 soh 야.

다음 세 가지 방법으로 결측을 처리하고 각각 성능을 비교해줘.
  A. 결측이 있는 행을 삭제
  B. 평균으로 채우기
  C. 다른 변수로 추정해서 채우기 (IterativeImputer)

성능은 RandomForestRegressor(n_estimators=200, random_state=42) 로
KFold(5, shuffle=True, random_state=42) 교차검증해서
MAE 와 R2 를 구해줘.

조건: 각 방법마다 학습에 쓰인 행 수도 함께 출력해줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>**Orange3 실습 ③에서 본 값과 비교하십시오.**

| 방법 | 행 수 | MAE |
|---|---|---|
| A · 행 삭제 | 149 | **0.234** |
| B · 평균 대체 | 168 | **0.250** |
| C · 모델 기반 | 168 | **0.244** |

세 방법의 차이가 거의 없습니다. **왜 그런지 설명할 수 있습니까?**</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate, KFold
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

F = ["cycle", "discharge_time", "max_temp", "mean_temp", "mean_voltage",
     "min_voltage", "re_ohm", "rct_ohm", "mean_current"]
b5 = df[df.cell_id == "B0005"].copy()
cv = KFold(5, shuffle=True, random_state=42)

def evaluate(data, name):
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    s = cross_validate(model, data[F], data.soh, cv=cv,
                       scoring=("neg_mean_absolute_error", "r2"))
    mae = -s["test_neg_mean_absolute_error"].mean()
    print(f"{name:14} 행 {len(data):3}  MAE {mae:.3f}  R2 {s['test_r2'].mean():.4f}")

# A. 행 삭제
evaluate(b5.dropna(subset=F), "A 행 삭제")

# B. 평균으로 채우기
tmp = b5.copy()
tmp[F] = tmp[F].fillna(tmp[F].mean())
evaluate(tmp, "B 평균 대체")

# C. 모델 기반
tmp = b5.copy()
tmp[F] = IterativeImputer(random_state=42, max_iter=20).fit_transform(tmp[F])
evaluate(tmp, "C 모델 기반")
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

### 실습 2-2 · 우회 누수 찾기

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">나는 배터리 SOH 예측에서 데이터 누수를 확인하고 있어.

soh = capacity / 2.0 * 100 으로 만든 값이라 capacity 는 입력에서 뺐어.
그런데도 성능이 너무 좋아서 다른 변수가 의심돼.

B0005 셀에서 다음 변수들을 각각 하나씩만 써서
LinearRegression 으로 soh 를 예측하고 R2 를 구해줘.
  capacity, discharge_time, mean_voltage, cycle

그리고 capacity 와 discharge_time 의 상관계수도 알려줘.

조건: 결과를 보고 어떤 변수가 누수인지 판단하는 근거도 설명해줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>**M2 실습 ④에서 본 값과 비교하십시오.**

| 변수 하나만 | 단독 R² |
|---|---|
| capacity | **1.00000** |
| mean_voltage | **0.98969** |
| discharge_time | **0.98234** |
| cycle | 0.97821 |

`capacity` ↔ `discharge_time` 상관 **+0.991**

**정전류 방전이라 방전시간이 용량과 같은 정보**입니다.</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
from sklearn.linear_model import LinearRegression

b5 = df[df.cell_id == "B0005"].dropna()

# 변수 하나씩 단독으로 예측
for col in ["capacity", "discharge_time", "mean_voltage", "cycle"]:
    X = b5[[col]]
    model = LinearRegression().fit(X, b5.soh)
    r2 = model.score(X, b5.soh)
    corr = b5[col].corr(b5.soh)
    print(f"{col:16} 단독 R2 {r2:.5f}  상관 {corr:+.4f}")

# 두 변수가 얼마나 같은지
print("\ncapacity <-> discharge_time 상관:",
      round(b5.capacity.corr(b5.discharge_time), 4))
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

---
# 3. 회귀 — M4 대조

**여기가 오늘의 핵심입니다.** 분할 방식에 따라 모델 순위가 뒤집히는 것을
파이썬에서도 확인합니다.

### 실습 3-1 · 모델 4종 비교 (무작위 교차검증)

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">나는 배터리 SOH 를 예측하는 모델을 비교하고 있어.
battery_cycles.csv 에서 B0005 셀만 쓸 거야.

입력은 운행 중 측정 가능한 값 7개만 쓸게.
  mean_voltage, mean_current, max_temp, mean_temp,
  min_voltage, re_ohm, rct_ohm
목표는 soh 야. 결측이 있는 행은 빼줘.

다음 네 모델을 KFold(5, shuffle=True, random_state=42) 로
교차검증해서 MAE 와 R2 를 비교해줘.
  1. LinearRegression
  2. Ridge(alpha=1.0)
  3. DecisionTreeRegressor(max_depth=4, random_state=42)
  4. RandomForestRegressor(n_estimators=200, random_state=42)

조건: 결과를 보기 좋게 표로 정리해줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>**M4 실험 ①에서 본 값과 비교하십시오.**

| 모델 | MAE | R² |
|---|---|---|
| Linear Regression | 0.418 | 0.9943 |
| Ridge (alpha=1) | 1.528 | 0.9510 |
| Decision Tree | 0.657 | 0.9912 |
| **Random Forest** | **0.381** | **0.9960** |

**Ridge가 유독 나쁩니다. 왜일까요?** (힌트 — M2에서 배운 것)</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate, KFold

F = ["mean_voltage", "mean_current", "max_temp", "mean_temp",
     "min_voltage", "re_ohm", "rct_ohm"]
d = df[df.cell_id == "B0005"].dropna(subset=F)
cv = KFold(5, shuffle=True, random_state=42)

models = [
    ("Linear Regression", LinearRegression()),
    ("Ridge (alpha=1)", Ridge(alpha=1.0)),
    ("Decision Tree", DecisionTreeRegressor(max_depth=4, random_state=42)),
    ("Random Forest", RandomForestRegressor(n_estimators=200, random_state=42)),
]

results = []
for name, model in models:
    s = cross_validate(model, d[F], d.soh, cv=cv,
                       scoring=("neg_mean_absolute_error", "r2"))
    results.append({
        "모델": name,
        "MAE": round(-s["test_neg_mean_absolute_error"].mean(), 3),
        "R2": round(s["test_r2"].mean(), 4),
    })

print(pd.DataFrame(results).to_string(index=False))
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

### 실습 3-2 · Ridge에 스케일링 적용

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">앞의 실험에서 Ridge 의 성능이 유독 나빴어 (MAE 1.528).
다른 모델은 0.4 정도인데 Ridge 만 3배 이상 나빠.

입력 변수들의 값 범위가 서로 많이 달라.
  discharge_time 은 수천 단위, re_ohm 은 0.04~0.06 정도야.

Ridge 에 StandardScaler 를 적용하기 전과 후를 비교해줘.
Pipeline 을 쓰면 좋겠어.

조건:
- 왜 Ridge 에만 스케일링이 필요한지 설명해줘
- 나무 계열에는 왜 필요 없는지도 알려줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>**M4-3에서 본 값과 비교하십시오.**

| | MAE | R² |
|---|---|---|
| 스케일 없음 | 1.528 | 0.9510 |
| **스케일 적용** | **0.430** | **0.9935** |

**3.5배 개선**됩니다. M2에서 배운 스케일링이 여기서 효과를 냅니다.</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 스케일 없음
s1 = cross_validate(Ridge(alpha=1.0), d[F], d.soh, cv=cv,
                    scoring=("neg_mean_absolute_error", "r2"))

# 스케일 적용 — Pipeline 으로 묶으면 학습 데이터 기준으로만 계산된다
pipe = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
s2 = cross_validate(pipe, d[F], d.soh, cv=cv,
                    scoring=("neg_mean_absolute_error", "r2"))

for tag, s in [("스케일 없음", s1), ("스케일 적용", s2)]:
    mae = -s["test_neg_mean_absolute_error"].mean()
    print(f"{tag:10}  MAE {mae:.3f}  R2 {s['test_r2'].mean():.4f}")
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

### 실습 3-3 · 시간순 분할 — 순위가 뒤집힙니다

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">나는 배터리 SOH 를 예측하는데, 미래 사이클을 맞히고 싶어.
무작위로 나누면 미래 데이터가 학습에 섞이니까
시간순으로 잘라서 평가하려고 해.

B0005 셀을 cycle 순서로 정렬한 뒤
앞 70% 로 학습하고 뒤 30% 를 예측해줘.

입력은 이 7개야.
  mean_voltage, mean_current, max_temp, mean_temp,
  min_voltage, re_ohm, rct_ohm

모델 네 개를 비교해줘.
  LinearRegression, Ridge(alpha=1.0),
  DecisionTreeRegressor(max_depth=4, random_state=42),
  RandomForestRegressor(n_estimators=200, random_state=42)

그리고 학습 구간과 평가 구간의 soh 범위도 각각 출력해줘.

조건: 결과가 예상과 다르게 나올 수 있는데, 그 이유도 설명해줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>**M4 실험 ②에서 본 값과 비교하십시오.**

| 모델 | R² |
|---|---|
| **Linear Regression** | **+0.567** |
| Ridge | −6.609 |
| Decision Tree | −5.916 |
| Random Forest | −5.938 |

**무작위에서 1등이던 Random Forest가 무너졌습니다.**

학습 구간 SOH 70.3~92.6% · 평가 구간 64.4~70.1% — **겹치지 않습니다.**</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
from sklearn.metrics import mean_absolute_error, r2_score

# 시간순 정렬 후 앞뒤로 자르기
d_sorted = d.sort_values("cycle").reset_index(drop=True)
k = int(len(d_sorted) * 0.7)
train, test = d_sorted[:k], d_sorted[k:]

print(f"학습 {len(train)}행  SOH {train.soh.min():.1f}~{train.soh.max():.1f}")
print(f"평가 {len(test)}행  SOH {test.soh.min():.1f}~{test.soh.max():.1f}")
print()

for name, model in models:
    model.fit(train[F], train.soh)
    pred = model.predict(test[F])
    mae = mean_absolute_error(test.soh, pred)
    r2 = r2_score(test.soh, pred)
    print(f"{name:20} MAE {mae:6.3f}  R2 {r2:+8.4f}")
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

### 왜 나무 계열이 무너지는지 눈으로 확인

숫자만으로는 원인을 알 수 없습니다. **예측선을 그려 봅니다.**

### 실습 3-4 · 예측선 그리기

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">앞의 시간순 분할 실험에서
Random Forest 의 R2 가 -5.94 로 무너졌어.
Linear Regression 은 +0.57 로 그나마 나았고.

왜 그런지 그래프로 확인하고 싶어.

x축을 cycle, y축을 soh 로 해서 다음을 한 그래프에 그려줘.
  1. 실제 soh 값 (전체 구간)
  2. Linear Regression 의 예측값 (평가 구간만)
  3. Random Forest 의 예측값 (평가 구간만)
  4. 학습과 평가가 나뉘는 지점에 세로 선

조건:
- matplotlib 을 쓰고 한글 대신 영문 라벨을 써줘
- 그래프를 보고 무엇을 알 수 있는지 설명해줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>**M4에서 본 그래프와 같은 모양이 나와야 합니다.**

- Random Forest 예측선이 **평평하게** 깔립니다
- Linear 예측선은 실제값을 **따라 내려갑니다**

**나무는 학습한 칸의 평균만 답하므로, 본 적 없는 낮은 값을 낼 수 없습니다.**</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
import matplotlib.pyplot as plt

# 두 모델 학습
lin = LinearRegression().fit(train[F], train.soh)
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(train[F], train.soh)

plt.figure(figsize=(12, 5))

# 실제값 (전체)
plt.plot(d_sorted.cycle, d_sorted.soh, color="gray", lw=1.5, label="Actual")

# 예측값 (평가 구간만)
plt.plot(test.cycle, lin.predict(test[F]), color="navy", lw=2.5, label="Linear")
plt.plot(test.cycle, rf.predict(test[F]), color="orangered", lw=2.5, label="Random Forest")

# 분할 지점
plt.axvline(train.cycle.max(), color="red", ls="--", alpha=0.6)
plt.text(train.cycle.max() + 2, 90, "prediction starts", color="red")

plt.xlabel("Cycle")
plt.ylabel("SOH (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.title("Random Forest cannot predict below its training range")
plt.show()
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

---
# 4. 저장한 모델 쓰기 — M4 대조

**Orange3에서 학습한 모델을 파이썬에서 그대로 씁니다.**

이것이 되면 Orange3는 연습용 도구가 아니라 실제 도구입니다.

### Orange3 설치

`.pkcls` 파일을 열려면 Orange 라이브러리가 필요합니다. **1~2분 걸립니다.**

In [ ]:
!pip install Orange3 -q
print("설치 완료")

### 실습 4-1 · 저장한 모델 불러와 예측하기

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">나는 Orange3 에서 Random Forest 모델을 학습해서
battery_rf.pkcls 파일로 저장했어. Colab 에 업로드했고
Orange3 라이브러리는 설치했어.

이 모델을 불러와서 예측에 쓰고 싶어.

데이터는 battery_cycles.csv 이고 B0005 셀만 쓸 거야.
입력 컬럼은 이 7개야.
  mean_voltage, mean_current, max_temp, mean_temp,
  min_voltage, re_ohm, rct_ohm

다음을 해줘.
  1. pickle 로 모델 파일을 불러오기
  2. 불러온 객체가 어떤 타입인지 확인
  3. 이 객체 안에 sklearn 모델이 들어 있는지 확인
  4. 그 sklearn 모델로 예측하고 앞 5개 값 출력
  5. 실제값과 비교해 MAE 계산

조건: 각 단계마다 주석을 달고, 오류가 날 만한 지점을 미리 알려줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>**확인할 것**

- 불러온 객체 타입은 `Orange.regression.random_forest.RandomForestRegressor`
- `model.skl_model` 에 sklearn 객체가 들어 있습니다
- **예측값이 Orange3 Predictions에서 본 것과 소수점까지 같아야 합니다**

모델 파일이 없다면 이 실습은 건너뛰십시오.</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
import pickle

# 1. 모델 불러오기
with open("battery_rf.pkcls", "rb") as f:
    model = pickle.load(f)

# 2. 어떤 객체인지 확인
print("타입:", type(model))

# 3. 안에 sklearn 모델이 들어 있다
sk = model.skl_model
print("내부 모델:", type(sk).__name__)
print("나무 개수:", sk.n_estimators)

# 4. 예측
d = df[df.cell_id == "B0005"].dropna(subset=F)
pred = sk.predict(d[F].values)
print("\n예측값 앞 5개:", np.round(pred[:5], 3))
print("실제값 앞 5개:", d.soh.values[:5])

# 5. 오차
from sklearn.metrics import mean_absolute_error
print("\nMAE:", round(mean_absolute_error(d.soh, pred), 4))
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

<table><tr><td style="background:#E4F3EB;border-left:4px solid #1F7A4D;padding:14px 18px">
<b>여기서 확인한 것</b><br><br>
Orange3에서 학습한 모델이 <b>sklearn 객체로 그대로 나옵니다.</b>
예측값도 소수점까지 같습니다.<br><br>
<b>Orange3에서 만든 결과물은 실제로 쓸 수 있습니다.</b>
</td></tr></table>

---
# 5. 분류 — M5 대조

숫자를 등급으로 바꿔 판정 문제로 만듭니다.

### 실습 5-1 · 2등급 분류와 혼동행렬

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">나는 배터리 SOH 로 양품/불량을 판정하는 모델을 만들고 있어.

battery_cycles.csv 에서 결측이 있는 행은 빼고 전체 셀을 쓸 거야.
입력은 이 7개야.
  mean_voltage, mean_current, max_temp, mean_temp,
  min_voltage, re_ohm, rct_ohm

soh 가 80 미만이면 불량(1), 80 이상이면 양품(0) 으로 등급을 만들어줘.

그 다음 RandomForestClassifier(n_estimators=100, random_state=42) 로
StratifiedKFold(5, shuffle=True, random_state=42) 교차검증을 하고
다음을 출력해줘.
  1. 혼동행렬 (TN, FP, FN, TP 각각 몇 개인지)
  2. 정확도, Precision, Recall, F1

조건:
- 불량을 양성(positive)으로 두고 계산해줘
- 각 지표가 혼동행렬의 어느 칸에서 나오는지 주석으로 설명해줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>**M5 실습 ⑬에서 본 값과 비교하십시오.**

| 항목 | 값 |
|---|---|
| TN · FP · FN · TP | 218 · 8 · **7** · 346 |
| 정확도 (CA) | 0.9741 |
| Precision | 0.9774 |
| **Recall** | **0.9802** |

**FN 7개** — 불량인데 양품으로 판정한 것. 배터리에서 가장 위험한 오류입니다.</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score)

d_all = df.dropna(subset=F)

# 등급 만들기 — 불량을 1(양성)로
y = (d_all.soh < 80).astype(int)
print("불량:", y.sum(), "· 양품:", (y == 0).sum())

# 교차검증 예측
cv_s = StratifiedKFold(5, shuffle=True, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
pred = cross_val_predict(clf, d_all[F], y, cv=cv_s)

# 혼동행렬 — 네 칸
tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
print(f"\nTN {tn} (양품을 양품으로)")
print(f"FP {fp} (양품을 불량으로 - 헛경보)")
print(f"FN {fn} (불량을 양품으로 - 가장 위험)")
print(f"TP {tp} (불량을 불량으로)")

# 지표 — 어느 칸에서 나오는지
print(f"\nCA        = (TP+TN)/전체 = {accuracy_score(y, pred):.4f}")
print(f"Precision = TP/(TP+FP)  = {precision_score(y, pred):.4f}")
print(f"Recall    = TP/(TP+FN)  = {recall_score(y, pred):.4f}")
print(f"F1                      = {f1_score(y, pred):.4f}")
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

### 실습 5-2 · 처음 보는 셀 판정하기

<table><tr><td style="background:#0E2A4A;color:#fff;padding:10px 14px;border-radius:8px 8px 0 0">
<b>STEP 1 · Gemini에게 요청하기</b></td></tr>
<tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:0 0 8px 8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">나는 배터리 불량 판정 모델을 만들었는데,
학습에 쓰지 않은 다른 셀도 제대로 판정하는지 확인하고 싶어.

battery_cycles.csv 에서
  학습 : B0005 와 B0006
  평가 : B0007
로 나눠줘. 결측 있는 행은 빼고.

입력은 앞과 같은 7개, 목표는 soh 80 미만이면 불량(1) 이야.

세 모델로 평가해줘.
  LogisticRegression (StandardScaler 와 Pipeline 으로)
  DecisionTreeClassifier(max_depth=4, random_state=42)
  RandomForestClassifier(n_estimators=100, random_state=42)

각각 정확도와 Recall, 그리고 놓친 불량 개수(FN)를 출력해줘.

조건: 결과를 보고 이 모델을 실제로 쓸 수 있을지 판단해줘</td></tr></table>

**위 프롬프트를 복사해** 왼쪽 아래 <b>Gemini</b> 아이콘을 눌러 붙여넣으십시오.
받은 코드를 <b>아래 빈 셀</b>에 붙여넣고 실행합니다.

In [ ]:
# ↓ Gemini가 준 코드를 여기에 붙여넣고 실행하십시오



<table><tr><td style="background:#FDEEE4;border-left:4px solid #EC6516;padding:12px 16px">
<b>STEP 2 · Orange3 결과와 대조</b><br><br>**M5-4에서 본 값과 비교하십시오.**

| 모델 | CA | Recall | 놓친 불량 |
|---|---|---|---|
| Logistic Regression | 0.470 | **0.000** | **79개** |
| Decision Tree | 0.510 | 0.076 | 73개 |
| Random Forest | 0.510 | 0.076 | 73개 |

**정확도 51%인데 불량 79개 중 73개를 놓쳤습니다.**

CA만 보면 "절반은 맞혔네"로 들리지만, 실제로는 쓸 수 없는 모델입니다.</td></tr></table>

결과가 다르다면 <b>설정이 다른 것</b>입니다. 무엇이 다른지 찾아보십시오.

<details>
<summary><b>막혔을 때만 펼치기</b> — 참고 코드 (복사해서 위 셀에 붙여넣기)</summary>

```
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

d_all = df.dropna(subset=F)

# 셀 단위로 나누기 — 평가 셀은 학습에 없어야 한다
train_c = d_all[d_all.cell_id.isin(["B0005", "B0006"])]
test_c = d_all[d_all.cell_id == "B0007"]

y_tr = (train_c.soh < 80).astype(int)
y_te = (test_c.soh < 80).astype(int)
print(f"학습 {len(train_c)}행 → 평가 {len(test_c)}행 (불량 {y_te.sum()}개)\n")

clfs = [
    ("Logistic Regression", make_pipeline(StandardScaler(),
                                          LogisticRegression(max_iter=1000))),
    ("Decision Tree", DecisionTreeClassifier(max_depth=4, random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=100, random_state=42)),
]

for name, clf in clfs:
    clf.fit(train_c[F], y_tr)
    p = clf.predict(test_c[F])
    tn, fp, fn, tp = confusion_matrix(y_te, p, labels=[0, 1]).ravel()
    ca = accuracy_score(y_te, p)
    rec = recall_score(y_te, p, zero_division=0)
    print(f"{name:22} CA {ca:.4f}  Recall {rec:.4f}  놓친 불량 {fn}개")
```

**이 코드를 그대로 쓰기 전에** 위의 Gemini 코드와 비교해 보십시오.
어디가 다르고, 왜 다른지 아는 것이 이 실습의 목적입니다.
</details>

---
# 6. 정리 — 내가 확인한 것

아래 표를 채우십시오. **Orange3와 파이썬 결과가 같았는지**가 핵심입니다.

| 실습 | Orange3 값 | 내 Colab 값 | 같은가 | 다르다면 원인 |
|---|---|---|---|---|
| 1-1 결측 개수 | 57 | | | |
| 2-1 결측 처리 A | MAE 0.234 | | | |
| 3-1 Random Forest | MAE 0.381 | | | |
| 3-2 Ridge 스케일 후 | MAE 0.430 | | | |
| 3-3 시간순 RF | R² −5.938 | | | |
| 4-1 저장 모델 예측 | — | | | |
| 5-1 혼동행렬 FN | 7개 | | | |
| 5-2 셀 단위 Recall | 0.076 | | | |

---

### 오늘 노트북에서 배운 것

**1. AI가 준 코드를 검증하는 법**

Gemini가 준 코드가 맞는지 어떻게 알았습니까?
**Orange3 결과와 대조**했기 때문입니다.
기준이 없으면 AI 답이 맞는지 알 수 없습니다.

**2. 파이썬을 몰라도 판단할 수 있다**

코드를 직접 쓰지 않았지만, **결과가 맞는지 판단**했습니다.
이것이 실무에서 더 중요한 능력입니다.

**3. 같은 분석을 두 도구로**

Orange3와 파이썬은 **같은 계산**을 합니다.
도구가 다를 뿐 원리는 같습니다.

---

### 마지막 프롬프트 — 오늘 전체 검토받기

<table><tr><td style="background:#061A30;color:#DCE6F2;padding:14px;border-radius:8px;
font-family:monospace;font-size:13px;white-space:pre-wrap">나는 배터리 SOH 데이터를 하루 동안 분석했어.

내가 한 것과 알아낸 것은 다음과 같아.

1. 전처리 : [내가 내린 결정과 근거]
2. 시각화 : [그래프에서 알아낸 것]
3. 회귀   : [고른 모델과 이유]
4. 분류   : [판정 결과와 한계]

다음을 검토해줘.
  1. 내 분석 흐름에서 논리적으로 빈 부분
  2. 결과를 보고서에 쓸 때 반드시 붙여야 할 조건
  3. 3일차에 스스로 분석할 때 주의할 점

조건: 칭찬보다 빈틈을 짚어줘. 내가 놓친 것을 알고 싶어.</td></tr></table>

---

### 3일차 준비

내일은 `battery_raw.csv`(**18.6만 행**)를 받아 **직접 요약**합니다.

| 오늘 배운 것 | 내일 쓸 곳 |
|---|---|
| Group by 집계 | 원본에서 사이클별 요약 만들기 |
| Formula 파생 변수 | 새 변수 설계 |
| 가설 세우기 | 스스로 문제 정의 |
| 결과 쓰는 법 | 4일차 발표 |

---

<div align="center">
<b>인공지능융합교육원 주식회사</b><br>
<sub>데이터 출처 — B. Saha and K. Goebel (2007), NASA Prognostics Data Repository</sub>
</div>